# Superstore Sales Analysis: SQL Analysis

**Goal:** answer the main business questions with SQL, using window functions, and confirm that the results match the pandas analysis.

**Input:** `data/processed/superstore_clean.csv` (output of notebook 01)
**Database:** SQLite file `data/processed/superstore.db`, rebuilt every time this notebook runs (it is not committed to GitHub).
**Queries:** the same 11 queries are saved in `sql/02_analysis_queries.sql`, and the last cell of this notebook runs them directly from that file.

**Note:** the dataset has no Profit, Quantity or Discount columns, so all queries are based on sales.

## 1. Load the cleaned data into SQLite

The cleaned CSV is loaded into a SQLite database as one table called `orders`.

- Column names are converted to snake_case (`Order Date` becomes `order_date`) so queries are simple to write.
- Dates are stored as `YYYY-MM-DD` text, which is how SQLite represents dates.
- `Postal Code` is read as text to keep leading zeros.
- `q()` is a small helper that runs a SQL query and returns the result as a DataFrame.

In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv("../data/processed/superstore_clean.csv",
                 parse_dates=["Order Date", "Ship Date"],
                 dtype={"Postal Code": str})
df.columns = [c.lower().replace("-", "_").replace(" ", "_") for c in df.columns]
for col in ["order_date", "ship_date"]:
    df[col] = df[col].dt.strftime("%Y-%m-%d")

con = sqlite3.connect("../data/processed/superstore.db")
df.to_sql("orders", con, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql(sql, con)

q("SELECT COUNT(*) AS rows FROM orders")      # expect 9799

,rows
0,9799


## 2. Queries

Each query has the business question, the SQL technique used, and the result.

### Q1. Headline KPIs

**Question:** how big is the business over the whole period?
**Technique:** aggregation with `COUNT(DISTINCT ...)`. Orders are counted with `DISTINCT order_id` because one order can span several rows.

**Result:** total sales 2,261,255.41, 4,922 orders, 793 customers and an average order value of 459.42, from 2015-01-03 to 2018-12-30. These match the pandas results.

In [2]:
q("""
SELECT ROUND(SUM(sales), 2) AS total_sales,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT customer_id) AS customers,
       ROUND(SUM(sales) / COUNT(DISTINCT order_id), 2) AS avg_order_value,
       MIN(order_date) AS first_order, MAX(order_date) AS last_order
FROM orders;
""")

,total_sales,orders,customers,avg_order_value,first_order,last_order
0,"2,261,255.41",4922,793,459.42,2015-01-03,2018-12-30


### Q2. Year-over-year growth

**Question:** is the business growing?
**Technique:** window function `LAG()` to read the previous year's sales on the same row, then compute the percentage change.

**Result:** sales fell 4.2% in 2016, then grew 30.6% in 2017 and 20.3% in 2018 (722,052). The first year has no growth value because there is no previous year.

In [3]:
q("""
WITH yearly AS (
    SELECT year, ROUND(SUM(sales), 2) AS sales FROM orders GROUP BY year
)
SELECT year, sales,
       LAG(sales) OVER (ORDER BY year) AS prev_year_sales,
       ROUND((sales - LAG(sales) OVER (ORDER BY year)) * 100.0
             / LAG(sales) OVER (ORDER BY year), 1) AS growth_pct
FROM yearly ORDER BY year;
""")

,year,sales,prev_year_sales,growth_pct
0,2015,"479,574.84",NaN,NaN
1,2016,"459,436.01","479,574.84",-4.20
2,2017,"600,192.55","459,436.01",30.60
3,2018,"722,052.02","600,192.55",20.30


### Q3. Seasonality: share and rank of each month

**Question:** which months matter most?
**Technique:** `SUM() OVER ()` for each month's share of total sales, and `RANK()` to order the months.

**Result:** September to December produce 51.8% of sales (13.3%, 8.8%, 15.5% and 14.2%), and November is ranked first. February (2.6%), January (4.2%) and April (6.0%) are the weakest months.

In [4]:
q("""
SELECT month, month_name,
       ROUND(SUM(sales), 2) AS sales,
       ROUND(SUM(sales) * 100.0 / SUM(SUM(sales)) OVER (), 1) AS share_pct,
       RANK() OVER (ORDER BY SUM(sales) DESC) AS sales_rank
FROM orders GROUP BY month, month_name ORDER BY month;
""")

,month,month_name,sales,share_pct,sales_rank
0,1,January,"94,291.63",4.20,11
1,2,February,"59,371.12",2.60,12
2,3,March,"197,573.59",8.70,5
3,4,April,"136,001.63",6.00,10
4,5,May,"154,086.72",6.80,7
5,6,June,"145,837.52",6.40,8
6,7,July,"145,535.69",6.40,9
7,8,August,"157,315.93",7.00,6
8,9,September,"300,103.41",13.30,3
9,10,October,"199,496.29",8.80,4


### Q4. Category share

**Question:** which product category sells the most?
**Technique:** `GROUP BY` with a share of the grand total using `SUM() OVER ()`.

**Result:** Technology 36.6%, Furniture 32.2% and Office Supplies 31.2%. The three categories are almost balanced.

In [5]:
q("""
SELECT category, ROUND(SUM(sales), 2) AS sales,
       ROUND(SUM(sales) * 100.0 / SUM(SUM(sales)) OVER (), 1) AS share_pct
FROM orders GROUP BY category ORDER BY sales DESC;
""")

,category,sales,share_pct
0,Technology,"827,455.87",36.60
1,Furniture,"728,377.20",32.20
2,Office Supplies,"705,422.33",31.20


### Q5. Top 3 sub-categories inside each category

**Question:** what are the leading products in each category?
**Technique:** `ROW_NUMBER() OVER (PARTITION BY category ...)` restarts the ranking inside each category; a CTE then keeps ranks 1 to 3.

**Result:** the leaders are Chairs (Furniture), Storage (Office Supplies) and Phones (Technology).

In [6]:
q("""
WITH ranked AS (
    SELECT category, sub_category, ROUND(SUM(sales), 2) AS sales,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY SUM(sales) DESC) AS rn
    FROM orders GROUP BY category, sub_category
)
SELECT category, sub_category, sales, rn AS rank_in_category
FROM ranked WHERE rn <= 3 ORDER BY category, rn;
""")

,category,sub_category,sales,rank_in_category
0,Furniture,Chairs,"322,541.36",1
1,Furniture,Tables,"202,810.63",2
2,Furniture,Bookcases,"113,813.20",3
3,Office Supplies,Storage,"219,343.39",1
4,Office Supplies,Binders,"200,028.79",2
5,Office Supplies,Appliances,"104,618.40",3
6,Technology,Phones,"327,782.45",1
7,Technology,Machines,"189,238.63",2
8,Technology,Accessories,"164,186.70",3


### Q6. Top 10 states with cumulative share

**Question:** how concentrated are sales geographically?
**Technique:** a running total `SUM(SUM(sales)) OVER (ORDER BY ...)` gives the cumulative share as states are added from largest to smallest.

**Result:** California (19.7%) and New York (13.5%) together produce 33.3% of sales, and the top 10 states produce 69.1%.

In [7]:
q("""
SELECT state, ROUND(SUM(sales), 2) AS sales,
       ROUND(SUM(sales) * 100.0 / SUM(SUM(sales)) OVER (), 1) AS share_pct,
       ROUND(SUM(SUM(sales)) OVER (ORDER BY SUM(sales) DESC) * 100.0
             / SUM(SUM(sales)) OVER (), 1) AS cumulative_share_pct
FROM orders GROUP BY state ORDER BY sales DESC LIMIT 10;
""")

,state,sales,share_pct,cumulative_share_pct
0,California,"446,306.46",19.70,19.70
1,New York,"306,361.15",13.50,33.30
2,Texas,"168,572.53",7.50,40.70
3,Washington,"135,206.85",6.00,46.70
4,Pennsylvania,"116,276.65",5.10,51.90
5,Florida,"88,436.53",3.90,55.80
6,Illinois,"79,236.52",3.50,59.30
7,Michigan,"76,136.07",3.40,62.60
8,Ohio,"74,848.98",3.30,66.00
9,Virginia,"70,636.72",3.10,69.10


### Q7. Segment sales by year

**Question:** how does each customer segment evolve?
**Technique:** pivot with `SUM(CASE WHEN year = ... THEN sales END)`, giving one column per year.

**Result:** Consumer is the largest segment (1.15M), then Corporate (688K) and Home Office (425K). All segments grew in 2017 and 2018; Corporate grew fastest, from 128K in 2015 to 236K in 2018.

In [8]:
q("""
SELECT segment,
       ROUND(SUM(CASE WHEN year = 2015 THEN sales END), 0) AS y2015,
       ROUND(SUM(CASE WHEN year = 2016 THEN sales END), 0) AS y2016,
       ROUND(SUM(CASE WHEN year = 2017 THEN sales END), 0) AS y2017,
       ROUND(SUM(CASE WHEN year = 2018 THEN sales END), 0) AS y2018,
       ROUND(SUM(sales), 0) AS total
FROM orders GROUP BY segment ORDER BY total DESC;
""")

,segment,y2015,y2016,y2017,y2018,total
0,Consumer,"262,957.00","265,356.00","291,143.00","328,604.00","1,148,061.00"
1,Corporate,"127,797.00","119,676.00","204,977.00","236,044.00","688,494.00"
2,Home Office,"88,821.00","74,404.00","104,072.00","157,404.00","424,701.00"


### Q8. Customer concentration by decile

**Question:** do a few customers produce most of the sales?
**Technique:** `NTILE(10)` splits the customers into ten equal groups by total sales.

**Result:** the top 10% of customers (80 customers) produce 31.2% of sales, the top 20% produce 48.7%, and the bottom 50% only about 19.4%. Revenue is concentrated, but not dependent on a handful of accounts.

*Note:* pandas gave 30.9% for the top 10% because it used 79 customers. `NTILE` puts 80 in the first group because 793 does not divide evenly by 10. The report uses 31%.

In [9]:
q("""
WITH cust AS (
    SELECT customer_id, SUM(sales) AS sales FROM orders GROUP BY customer_id
),
bucketed AS (
    SELECT *, NTILE(10) OVER (ORDER BY sales DESC) AS decile FROM cust
)
SELECT decile, COUNT(*) AS customers, ROUND(SUM(sales), 2) AS sales,
       ROUND(SUM(sales) * 100.0 / SUM(SUM(sales)) OVER (), 1) AS share_pct
FROM bucketed GROUP BY decile ORDER BY decile;
""")

,decile,customers,sales,share_pct
0,1,80,"704,697.12",31.20
1,2,80,"395,701.46",17.50
2,3,80,"294,217.12",13.00
3,4,79,"234,275.07",10.40
4,5,79,"191,993.66",8.50
5,6,79,"155,710.35",6.90
6,7,79,"117,409.21",5.20
7,8,79,"86,852.74",3.80
8,9,79,"59,542.13",2.60
9,10,79,"20,856.56",0.90


### Q9. Repeat customers

**Question:** do customers come back?
**Technique:** a CTE counts distinct orders per customer, then `CASE` flags customers with more than one order.

**Result:** 780 of 793 customers (98.4%) ordered more than once, with 6.2 orders per customer on average. Customer loyalty is very high.

In [10]:
q("""
WITH per_customer AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS orders
    FROM orders GROUP BY customer_id
)
SELECT COUNT(*) AS customers,
       SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END) AS repeat_customers,
       ROUND(SUM(CASE WHEN orders > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS repeat_rate_pct,
       ROUND(AVG(orders), 1) AS avg_orders_per_customer
FROM per_customer;
""")

,customers,repeat_customers,repeat_rate_pct,avg_orders_per_customer
0,793,780,98.40,6.20


### Q10. Shipping performance

**Question:** how fast is each ship mode and how much is it used?
**Technique:** `AVG(shipping_days)` and share of sales per ship mode.

**Result:** Standard Class is used the most (59.3% of sales, 5.01 days on average). Second Class takes 3.25 days, First Class 2.18 and Same Day 0.04, so each mode matches its promise.

In [11]:
q("""
SELECT ship_mode, COUNT(*) AS order_lines,
       ROUND(AVG(shipping_days), 2) AS avg_days,
       ROUND(SUM(sales), 2) AS sales,
       ROUND(SUM(sales) * 100.0 / SUM(SUM(sales)) OVER (), 1) AS share_pct
FROM orders GROUP BY ship_mode ORDER BY sales DESC;
""")

,ship_mode,order_lines,avg_days,sales,share_pct
0,Standard Class,5858,5.01,"1,340,549.94",59.30
1,Second Class,1902,3.25,"449,914.18",19.90
2,First Class,1501,2.18,"345,572.26",15.30
3,Same Day,538,0.04,"125,219.04",5.50


### Q11. Running total of sales in 2018

**Question:** how does the latest year build up month by month?
**Technique:** `SUM() OVER (PARTITION BY year ORDER BY month)` gives the cumulative sales inside the year.

**Result:** sales reach 722,052 by December; November alone adds about 118K, the largest monthly sales of the whole period.

In [12]:
q("""
WITH m AS (
    SELECT year, month, SUM(sales) AS sales FROM orders GROUP BY year, month
)
SELECT year, month, ROUND(sales, 0) AS sales,
       ROUND(SUM(sales) OVER (PARTITION BY year ORDER BY month), 0) AS running_total
FROM m WHERE year = 2018 ORDER BY month;
""")

,year,month,sales,running_total
0,2018,1,"43,476.00","43,476.00"
1,2018,2,"19,921.00","63,397.00"
2,2018,3,"58,863.00","122,261.00"
3,2018,4,"35,542.00","157,803.00"
4,2018,5,"43,826.00","201,629.00"
5,2018,6,"48,191.00","249,820.00"
6,2018,7,"44,825.00","294,645.00"
7,2018,8,"62,838.00","357,482.00"
8,2018,9,"86,153.00","443,635.00"
9,2018,10,"77,448.00","521,083.00"


## 3. Check: run the queries from the `.sql` file

The queries above were typed in this notebook. This cell reads `sql/02_analysis_queries.sql`, splits it into its 11 queries (each starts with a `-- Q` comment) and runs every one, so we confirm that the submitted `.sql` file works exactly as written.

**Expected output:** 11 lines, one per query, with no error.

In [13]:
import re

with open("../sql/02_analysis_queries.sql", encoding="utf-8") as f:
    text = f.read()

blocks = re.split(r"(?m)^(?=-- Q\d+\.)", text)[1:]
for b in blocks:
    title = b.split("\n")[0]
    result = q(b)
    print(f"{title}  ->  {result.shape[0]} rows x {result.shape[1]} columns")

-- Q1. Headline KPIs  ->  1 rows x 6 columns
-- Q2. Year-over-year growth (window function LAG)  ->  4 rows x 4 columns
-- Q3. Seasonality: share and rank of each month (RANK)  ->  12 rows x 5 columns
-- Q4. Category share  ->  3 rows x 3 columns
-- Q5. Top 3 sub-categories inside each category (ROW_NUMBER + PARTITION BY)  ->  9 rows x 4 columns
-- Q6. Top 10 states with cumulative share (running total window)  ->  10 rows x 4 columns
-- Q7. Segment sales by year (pivot with CASE)  ->  3 rows x 6 columns
-- Q8. Customer concentration by decile (NTILE)  ->  10 rows x 4 columns
-- Q9. Repeat customers  ->  1 rows x 4 columns
-- Q10. Shipping performance  ->  4 rows x 5 columns
-- Q11. Running total of sales in 2018 (cumulative window)  ->  12 rows x 4 columns


## Summary

- All key figures (total sales, orders, customers, average order value, yearly growth, state shares, shipping averages) match the pandas analysis, which confirms the results.
- SQL techniques used: aggregation, CTEs, window functions (`LAG`, `RANK`, `ROW_NUMBER`, `NTILE`, running totals), conditional aggregation (`CASE`) and `PARTITION BY`.
- The business findings from these queries feed the insights and recommendations in the final report.

In [14]:
con.close()